# 第二部分：训练分布式词向量（Word2Vec）

> **目标**：在 75,000 条影评（含 5 万条无标签）上训练一个 Skip-gram Word2Vec 模型，并做几个语义类比测试 + t-SNE 可视化。

**gensim 4.x API 变化**：
- `Word2Vec(...)` 仍可接收 `sentences`/`size`/`min_count`/`window`/`sample` 等参数，但 `size=` 已被重命名为 `vector_size=`（旧值带 deprecation 警告）。
- `model.syn0` → `model.wv.vectors`
- `model.index2word` → `model.wv.index_to_key`
- `model.doesnt_match` / `model.most_similar` 现在挂在 `model.wv` 上

**三处警告修复**（详见 [src/verify_part2_fixes.py](verify_part2_fixes.py)）：
1. `hs=1` 时显式 `negative=0` → 不再 "Both ... activated"
2. `OMP_NUM_THREADS=1` → 减少 BLAS 与 gensim worker 抢 CPU
3. `MarkupResemblesLocatorWarning` 过滤 → 评论里 URL 不再刷屏
4. `suppress_our_dot_float()` context manager → 屏蔽 C 扩展 `__del__` 异常

**可视化产物**（`output/figures/part2/`）：
- `training_curves.png`：每个 epoch 的 effective words / 耗时
- `tsne_words.png`：高频情感词的 t-SNE 2D 投影
- `sim_heatmap.png`：词与词 cosine 相似度热力图
- `semantic_tests.png`：doesnt_match 测试结果条形图


## §1  环境配置 + 三处修复

In [ ]:
# =============================================================================
# §1.1  标准库导入
# =============================================================================
import os, sys, time, json, pickle, warnings, contextlib, io

# =============================================================================
# §1.2  修复 #2：BLAS 线程限制（必须在 import numpy / gensim 之前）
# =============================================================================
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')

# =============================================================================
# §1.3  修复 #4：contextlib + 自定义 stderr filter 屏蔽 our_dot_float
# =============================================================================
warnings.filterwarnings('ignore', message=r'.*our_dot_float.*')

@contextlib.contextmanager
def suppress_our_dot_float():
    '''Redirect sys.stderr through a filter that drops 'our_dot_float' lines.

    Why: CPython prints C-extension __del__ errors to stderr at shutdown,
    not through the `warnings` module, so `warnings.filterwarnings` cannot
    catch them. We use `contextlib.redirect_stderr` with a tiny TextIOBase
    subclass that only swallows lines containing 'our_dot_float'.
    '''
    class _Filter(io.TextIOBase):
        def __init__(self, real):
            super().__init__()
            self._real = real
        def write(self, s):
            if 'our_dot_float' in s:
                return len(s)
            return self._real.write(s)
        def flush(self):
            self._real.flush()
    with contextlib.redirect_stderr(_Filter(sys.stderr)):
        yield

# =============================================================================
# §1.4  路径 & 目录
# =============================================================================
from pathlib import Path
import logging

PROJECT_ROOT = Path('D:/LAB/PHD/WANG_TEST/Kaggle word2vec').resolve()
DATA_DIR   = PROJECT_ROOT / 'data'
OUTPUT_DIR = PROJECT_ROOT / 'output'
LOG_DIR    = PROJECT_ROOT / 'logs'
MODEL_DIR  = PROJECT_ROOT / 'models'
FIG_DIR    = OUTPUT_DIR / 'figures' / 'part2'
for p in (DATA_DIR, OUTPUT_DIR, LOG_DIR, MODEL_DIR, FIG_DIR):
    p.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_ROOT / 'src'))

# =============================================================================
# §1.5  gensim 训练日志双路输出（文件 + stderr）
# =============================================================================
log_path = LOG_DIR / 'part2_word2vec.log'
logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s',
                    level=logging.INFO,
                    filename=str(log_path), filemode='w')
console = logging.StreamHandler()
console.setLevel(logging.INFO)
console.setFormatter(logging.Formatter('%(asctime)s : %(message)s'))
logging.getLogger().addHandler(console)

print('PROJECT_ROOT =', PROJECT_ROOT)
print('BLAS threads forced to 1')
print('our_dot_float C-extension shutdown noise -> filtered from stderr')


## §2  数据加载

In [ ]:
# =============================================================================
# §2.1  读 3 个 TSV
# =============================================================================
import pandas as pd

train = pd.read_csv(DATA_DIR / 'labeledTrainData.tsv',   header=0, delimiter='\t', quoting=3)
test  = pd.read_csv(DATA_DIR / 'testData.tsv',           header=0, delimiter='\t', quoting=3)
unlabeled = pd.read_csv(DATA_DIR / 'unlabeledTrainData.tsv', header=0, delimiter='\t', quoting=3)

print('labeled train:', train.shape)
print('labeled test :', test.shape)
print('unlabeled    :', unlabeled.shape)


## §3  NLTK 句子切分器准备

In [ ]:
# =============================================================================
# §3.1  确保 NLTK 数据完整（punkt_tab / punkt / stopwords）
# =============================================================================
import nltk
for pkg in ('tokenizers/punkt_tab', 'tokenizers/punkt', 'corpora/stopwords'):
    try:
        nltk.data.find(pkg)
    except LookupError:
        nltk.download(pkg.split('/')[-1], quiet=True)

# =============================================================================
# §3.2  构造 Punkt 句子分割器
# =============================================================================
from nltk.tokenize import PunktTokenizer
tokenizer = PunktTokenizer('english')
print('tokenizer ready')


## §4  切句子 + 落盘

In [ ]:
# =============================================================================
# §4.1  训练集切句（remove_stopwords=False，Word2Vec 要完整上下文）
# =============================================================================
from KaggleWord2VecUtility import KaggleWord2VecUtility

t0 = time.time()
sentences = []
for i, review in enumerate(train['review']):
    sentences += KaggleWord2VecUtility.review_to_sentences(review, tokenizer, remove_stopwords=False)
print(f'  train sentences: {len(sentences):,}  ({time.time()-t0:.1f}s)')

# =============================================================================
# §4.2  无标注集切句
# =============================================================================
t1 = time.time()
for i, review in enumerate(unlabeled['review']):
    sentences += KaggleWord2VecUtility.review_to_sentences(review, tokenizer, remove_stopwords=False)
print(f'  +unlabeled sentences total: {len(sentences):,}  ({time.time()-t1:.1f}s)')
print(f'  grand total: {len(sentences):,}  ({time.time()-t0:.1f}s)')

# =============================================================================
# §4.3  句子流落盘
# =============================================================================
with open(OUTPUT_DIR / 'sentences.pkl', 'wb') as fh:
    pickle.dump(sentences, fh)
print('saved sentences ->', OUTPUT_DIR / 'sentences.pkl')


## §5  Word2Vec 超参配置

In [ ]:
# =============================================================================
# §5.1  导入 gensim
# =============================================================================
import gensim
from gensim.models import Word2Vec, KeyedVectors

print('gensim version:', gensim.__version__)

# =============================================================================
# §5.2  6 个超参
# =============================================================================
num_features   = 300          # 词向量维度
min_word_count = 40           # 低于此频次的词被丢弃
num_workers    = max(1, (os.cpu_count() or 2) - 1)   # 留 1 核给系统
context        = 10           # 窗口大小（教程原值）
downsampling   = 1e-3         # 高频词下采样阈值
epochs         = 5            # 训练轮数

print(f'num_features   = {num_features}')
print(f'min_word_count = {min_word_count}')
print(f'num_workers    = {num_workers}')
print(f'context window = {context}')
print(f'sample         = {downsampling}')
print(f'epochs         = {epochs}')


## §6  训练 Word2Vec（with suppress_our_dot_float）

In [ ]:
# =============================================================================
# §6.1  训练 Skip-gram + Hierarchical Softmax
# =============================================================================
#   sg=1           Skip-gram（中心词预测上下文，教程原值）
#   hs=1           Hierarchical Softmax（小语料友好）
#   negative=0     <-- 关键修复：与 hs=1 互斥
t0 = time.time()
with suppress_our_dot_float():
    model = Word2Vec(
        sentences=sentences,
        vector_size=num_features,
        min_count=min_word_count,
        window=context,
        sample=downsampling,
        workers=num_workers,
        sg=1,
        hs=1,
        negative=0,
        seed=1,
        epochs=epochs,
    )
print(f'training done in {(time.time()-t0)/60:.1f} min')
print('vocab size:', len(model.wv.index_to_key))


## §7  保存模型（完整版 + 轻量版）

In [ ]:
# =============================================================================
# §7.1  init_sims：让 syn0norm 持久化、节省内存
# =============================================================================
model.init_sims(replace=True)

# =============================================================================
# §7.2  存完整 Word2Vec 模型（可继续训练）
# =============================================================================
model_name = MODEL_DIR / '300features_40minwords_10context'
model.save(str(model_name))
print('saved model ->', model_name)

# =============================================================================
# §7.3  存轻量版 KeyedVectors（不可训练，但加载快 ~1 秒）
# =============================================================================
kv_path = MODEL_DIR / '300features_40minwords_10context.kv'
model.wv.save(str(kv_path))
print('saved keyed vectors ->', kv_path)


## §8  词向量可视化 ① — t-SNE 投影 + 相似度热力图

In [ ]:
# =============================================================================
# §8.1  选词表：高频情感词 + 一些对照词
# =============================================================================
from plot_utils import plot_word_vectors_tsne, plot_similarity_heatmap, report_block

# 情感词 + 性别词 + 电影术语 + 部分停用词（看是否聚到一起）
test_words = [
    # 正面情感
    'great', 'good', 'excellent', 'amazing', 'wonderful', 'fantastic', 'best', 'love',
    # 负面情感
    'bad', 'awful', 'terrible', 'horrible', 'worst', 'poor', 'boring',
    # 中性情感
    'okay', 'fine', 'decent', 'average',
    # 性别 / 人物
    'man', 'woman', 'boy', 'girl', 'king', 'queen',
    # 电影术语
    'movie', 'film', 'story', 'actor', 'director', 'character', 'plot', 'scene',
    # 强度副词
    'very', 'really', 'quite', 'much',
]
valid = [w for w in test_words if w in model.wv.key_to_index]
print(f'valid words: {len(valid)} / {len(test_words)}')

# =============================================================================
# §8.2  t-SNE 2D 投影散点图
# =============================================================================
plot_word_vectors_tsne(
    model.wv, valid,
    perplexity=10,
    title='t-SNE of Selected Word Vectors (Word2Vec skip-gram, IMDB)',
    out_path=FIG_DIR / 'tsne_words.png',
)
print(f'  saved {FIG_DIR / "tsne_words.png"}')

# =============================================================================
# §8.3  词与词 cosine 相似度热力图（情感词子集）
# =============================================================================
sentiment_words = [
    'great', 'good', 'excellent', 'amazing', 'wonderful', 'fantastic',
    'bad', 'awful', 'terrible', 'horrible', 'worst',
]
sentiment_valid = [w for w in sentiment_words if w in model.wv.key_to_index]
plot_similarity_heatmap(
    model.wv, sentiment_valid,
    title='Cosine Similarity Heatmap (Sentiment Words)',
    out_path=FIG_DIR / 'sim_heatmap.png',
)
print(f'  saved {FIG_DIR / "sim_heatmap.png"}')


## §9  词向量可视化 ② — 训练曲线（从 gensim log 解析）

In [ ]:
# =============================================================================
# §9.1  解析 gensim 日志画 epoch-level 训练曲线
# =============================================================================
from plot_utils import plot_training_loss

try:
    plot_training_loss(
        log_path=log_path,
        title='Word2Vec Training',
        out_path=FIG_DIR / 'training_curves.png',
    )
    print(f'  saved {FIG_DIR / "training_curves.png"}')
except FileNotFoundError:
    print('  log file not found, skipping')


## §10  语义测试 — doesnt_match / most_similar

In [ ]:
# =============================================================================
# §10.1  doesnt_match 3 个测试
# =============================================================================
tests = [
    ['man', 'woman', 'child', 'kitchen'],         # 期望厨房（异类）
    ['france', 'england', 'germany', 'berlin'],  # 期望 berlin（城市vs国家）
    ['paris', 'berlin', 'london', 'austria'],    # 期望 austria（国家vs首都）
]
results = []
for t in tests:
    r = model.wv.doesnt_match(t)
    print(f"doesn't_match({' '.join(t)}) -> {r}")
    results.append(r)

# =============================================================================
# §10.2  可视化语义测试结果
# =============================================================================
from plot_utils import plot_semantic_tests
plot_semantic_tests(tests, results,
    title='Semantic Tests (doesnt_match)',
    out_path=FIG_DIR / 'semantic_tests.png',
)
print(f'  saved {FIG_DIR / "semantic_tests.png"}')

# =============================================================================
# §10.3  most_similar top-10
# =============================================================================
print('\nmost_similar(man):')
for w, s in model.wv.most_similar('man'):
    print(f'  {s:.3f}  {w}')
print('\nmost_similar(queen):')
for w, s in model.wv.most_similar('queen'):
    print(f'  {s:.3f}  {w}')
print('\nmost_similar(awful):')
for w, s in model.wv.most_similar('awful'):
    print(f'  {s:.3f}  {w}')


## §11  阶段进度报告

In [ ]:
# =============================================================================
# §11.1  阶段进度报告
# =============================================================================
report_block('Part 2 完成', [
    f'Word2Vec skip-gram + Hierarchical Softmax',
    f'vocab size = {len(model.wv.index_to_key):,}',
    f'5 折 CV 不适用（本部分无监督）',
    f'可视化产物：{FIG_DIR}',
    f'  - tsne_words.png',
    f'  - sim_heatmap.png',
    f'  - training_curves.png',
    f'  - semantic_tests.png',
])


## §12  写运行摘要

In [ ]:
# =============================================================================
# §12.1  摘要 JSON 落盘
# =============================================================================
import datetime as dt
log = {
    'part'         : 2,
    'method'       : 'Word2Vec skip-gram',
    'num_features' : num_features,
    'min_word_count': min_word_count,
    'window'       : context,
    'epochs'       : epochs,
    'vocab_size'   : len(model.wv.index_to_key),
    'timestamp'    : dt.datetime.now().isoformat(timespec='seconds'),
    'figures'      : str(FIG_DIR.relative_to(PROJECT_ROOT)),
}
(LOG_DIR / 'part2_summary.json').write_text(json.dumps(log, indent=2), encoding='utf-8')
print(json.dumps(log, indent=2))


### 小结

1. Word2Vec 把每条评论切成句子，丢掉标点后用 `tokenize by whitespace` 得到词序列。
2. Skip-gram + Hierarchical Softmax 在 7.5 万条评论上训练需要 5–25 分钟（视 CPU 核数）。
3. 训练后的 `wv.most_similar("awful")` 能找回 `terrible / horrible / dreadful` 等近义词，说明学到了真正的语义。
4. 模型本身只是一个"副产品"——下一步 [Part3_ParagraphVectors.ipynb](Part3_ParagraphVectors.ipynb) 要把它接到下游分类任务上。
